# 🛡️ Multimodal Deepfake Detection on LAV-DF (RMS-AV & Baselines)
### Localized Audio-Visual DeepFake Dataset Training & Research Pipeline
**Author:** Research Team | **Target Dataset:** LAV-DF (KaggleHub 23.1 GB)  
**Proposed Method:** RMS-AV (Reliability- and Manipulation-Aware Selective Audio-Visual Fusion)

This notebook provides a step-by-step, reproducible workflow to:
1. Verify the downloaded LAV-DF directory and inspect annotation schemas.
2. Confirm binary labels (Real vs Fake) and temporal manipulation boundaries (`fake_periods`).
3. Audit cross-split source separation to guarantee zero data leakage.
4. Train unimodal baselines (Audio, Video), Standard Fusion, and RMS-AV.
5. Evaluate on the held-out test split under clean and modality-drop stress testing.
6. Persist trained checkpoints and evaluation metrics to Google Drive.


---
## Stage 0: GPU Environment Setup & Verification
Run this cell first to ensure a CUDA-capable GPU (T4, V100, or A100) is allocated.


In [ ]:
# Cell 0.1: Check GPU acceleration
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")


### Cell 0.2: Clone Repository & Install Dependencies
Clone the multimodal deepfake detection repository and install required packages.


In [ ]:
# Cell 0.2: Setup project workspace and install dependencies
import os

# If running directly inside a cloned repo directory, skip git clone
if not os.path.exists("lavdf_pipeline"):
    !git clone https://github.com/akhildragon-07/DEEP_LEARNING.git
    %cd DEEP_LEARNING

# Install requirements
!pip install -q -r lavdf_pipeline/requirements.txt
print("Dependencies installed successfully.")


---
## Stage 1: Dataset Inspection & Schema Verification (MANDATORY FIRST STEP)
> **IMPORTANT:** Do NOT start training until running this section. We must verify:
> 1. The exact location of the 23.1 GB LAV-DF download.
> 2. Whether files are compressed (`.zip` / `.tar`) or already extracted.
> 3. The exact schema of `metadata.min.json` (or annotations).
> 4. Binary real/fake counts and modality manipulation breakdown.


In [ ]:
# Cell 1.1: Verify dataset path and top-level contents
import os
import glob

# Default KaggleHub path in Google Colab
DATASET_DIR = "/root/.cache/kagglehub/datasets/elin75/localized-audio-visual-deepfake-dataset-lav-df/versions/1"

# Check if directory exists
if os.path.exists(DATASET_DIR):
    print(f"✓ Found LAV-DF directory at: {DATASET_DIR}")
    entries = os.listdir(DATASET_DIR)
    print(f"Total entries in directory: {len(entries)}")
    for e in entries[:15]:
        p = os.path.join(DATASET_DIR, e)
        sz = os.path.getsize(p) if os.path.isfile(p) else "DIR"
        print(f"  - {e} ({sz})")
else:
    print(f"✗ Directory NOT found: {DATASET_DIR}")
    print("If you downloaded to another folder or mounted Drive, update DATASET_DIR below.")


In [ ]:
# Cell 1.2: Check for unextracted archive files and extract if necessary
import zipfile
import tarfile

archives = glob.glob(os.path.join(DATASET_DIR, "*.zip")) + glob.glob(os.path.join(DATASET_DIR, "*.tar.gz"))
if archives:
    print(f"Detected {len(archives)} archive file(s) that may need extraction:")
    for a in archives:
        sz_gb = os.path.getsize(a) / (1024**3)
        print(f"  - {os.path.basename(a)} ({sz_gb:.2f} GB)")
        
    # Unpack if extracted data directory is not yet present
    extract_target = os.path.join(DATASET_DIR, "extracted")
    # Uncomment to extract if needed:
    # os.makedirs(extract_target, exist_ok=True)
    # with zipfile.ZipFile(archives[0], 'r') as zip_ref:
    #     zip_ref.extractall(extract_target)
    # print("Extraction complete.")
else:
    print("No compressed archives found; dataset appears extracted.")


In [ ]:
# Cell 1.3: Run the comprehensive LAV-DF dataset audit script
!python inspect_lavdf.py --data_dir "$DATASET_DIR" --output_dir results_lavdf


In [ ]:
# Cell 1.4: Display audit report summary
from IPython.display import Markdown, display

report_md_path = "results_lavdf/LAVDF_AUDIT_REPORT.md"
if os.path.exists(report_md_path):
    with open(report_md_path, "r", encoding="utf-8") as f:
        display(Markdown(f.read()))
else:
    print("Audit report not found. Please verify Cell 1.3 output.")


---
## Stage 2: Pipeline Validation & Unit Tests
Run the unit test suite on the GPU instance to verify that models, reliability gating, validity masking, and preprocessors execute without tensor shape errors.


In [ ]:
# Cell 2.1: Execute test suite
!python -m unittest test_lavdf_pipeline.py


---
## Stage 3: Model Training (Small Stages)
We train four model variants on identical data splits for rigorous comparison:
1. **Audio Baseline:** Acoustic-only 2D CNN encoder on 80-channel log-mel spectrograms.
2. **Video Baseline:** Visual-only ResNet18 encoder with temporal pooling on uniform video frames.
3. **Standard Fusion:** Direct feature concatenation $[v; a]$ with a joint classifier head.
4. **RMS-AV (Proposed):** Dynamic reliability gate with validity masking and auxiliary unimodal supervision.

> **Configuration:** Adjust `epochs`, `batch_size`, and `METADATA_PATH` as reported in Stage 1.


In [ ]:
# Cell 3.0: Set paths based on Stage 1 audit findings
import json

with open("results_lavdf/lavdf_inspection_report.json") as f:
    audit_data = json.load(f)

METADATA_PATH = audit_data["metadata_audit"]["metadata_file"]
MEDIA_ROOT = audit_data["metadata_audit"]["verified_media_base"]

print(f"Using Metadata:   {METADATA_PATH}")
print(f"Using Media Root: {MEDIA_ROOT}")


In [ ]:
# Cell 3.1: Train Audio Baseline
!python -m lavdf_pipeline.train \
    --mode audio \
    --metadata_path "$METADATA_PATH" \
    --media_root "$MEDIA_ROOT" \
    --batch_size 16 \
    --epochs 10 \
    --lr 0.0001 \
    --patience 4


In [ ]:
# Cell 3.2: Train Video Baseline
!python -m lavdf_pipeline.train \
    --mode video \
    --metadata_path "$METADATA_PATH" \
    --media_root "$MEDIA_ROOT" \
    --batch_size 16 \
    --epochs 10 \
    --lr 0.0001 \
    --patience 4


In [ ]:
# Cell 3.3: Train Standard Fusion Baseline
!python -m lavdf_pipeline.train \
    --mode fusion \
    --metadata_path "$METADATA_PATH" \
    --media_root "$MEDIA_ROOT" \
    --batch_size 16 \
    --epochs 10 \
    --lr 0.0001 \
    --patience 4


In [ ]:
# Cell 3.4: Train Proposed RMS-AV Model
!python -m lavdf_pipeline.train \
    --mode rmsav \
    --metadata_path "$METADATA_PATH" \
    --media_root "$MEDIA_ROOT" \
    --batch_size 16 \
    --epochs 10 \
    --lr 0.0001 \
    --aux_weight 0.20 \
    --patience 4


---
## Stage 4: Held-Out Benchmark & Stress Testing
Evaluate all four trained models on the held-out test split under:
1. **Clean:** Normal audio-visual input.
2. **Missing Video:** Video stream dropped/masked.
3. **Missing Audio:** Audio stream dropped/masked.

This empirically evaluates how well the RMS-AV dynamic reliability gate suppresses missing modalities compared to standard concatenation.


In [ ]:
# Cell 4.1: Run benchmark evaluation across all models and conditions
!python -m lavdf_pipeline.evaluate \
    --metadata_path "$METADATA_PATH" \
    --media_root "$MEDIA_ROOT" \
    --checkpoint_dir checkpoints_lavdf \
    --output_dir results_lavdf


In [ ]:
# Cell 4.2: Display benchmark comparison table
import pandas as pd

bench_csv = "results_lavdf/lavdf_benchmark_comparison.csv"
if os.path.exists(bench_csv):
    df_results = pd.read_csv(bench_csv)
    display(df_results)
else:
    print("Benchmark comparison file not found.")


---
## Stage 5: Interactive Forensic Inference
Run single-video forensic inference on any sample clip. Displays manipulation probability, logit, uncertainty margin, and reliability gate weights.


In [ ]:
# Cell 5.1: Test inference engine
from lavdf_pipeline.inference import LAVDFInferenceEngine
import glob

ckpt_rmsav = "checkpoints_lavdf/lavdf_rmsav_best.pth"
if os.path.exists(ckpt_rmsav):
    engine = LAVDFInferenceEngine(checkpoint_path=ckpt_rmsav, mode="rmsav")
    
    # Pick a sample video from dataset
    sample_videos = glob.glob(os.path.join(MEDIA_ROOT, "**/*.mp4"), recursive=True)
    if sample_videos:
        test_clip = sample_videos[0]
        print(f"Testing on sample: {test_clip}")
        result = engine.predict_video(test_clip, threshold=0.50, uncertainty_margin=0.05)
        print(json.dumps(result, indent=2))
else:
    print("RMS-AV checkpoint not found. Complete Stage 3 first.")


---
## Stage 6: Checkpoint Persistence & Google Drive Backup
Save your trained checkpoints, training logs, and benchmark evaluation tables to your Google Drive to ensure they are preserved when the Colab session ends.


In [ ]:
# Cell 6.1: Mount Drive and copy checkpoints
from google.colab import drive
import shutil

drive.mount('/content/drive')

drive_backup_dir = "/content/drive/MyDrive/RMS_AV_LAVDF_Checkpoints"
os.makedirs(drive_backup_dir, exist_ok=True)

# Copy checkpoints and results
!cp -r checkpoints_lavdf/* "$drive_backup_dir/"
!cp -r results_lavdf/* "$drive_backup_dir/"

print(f"✓ All checkpoints and results successfully backed up to: {drive_backup_dir}")
